In [ ]:
"""
Build Pyrome-specific FlamMap lookups

author: maxwell.cook@colostate.edu
"""

import os, sys

from pathlib import Path
from os.path import join
from fb_tools.weather import (
    load_gridmet_csv, build_flammap_scenario_cache,
    load_flammap_scenario_cache
)

# use the current working directory
projdir = Path.cwd().parents[1] # moves up two, outside code directory
print(f"Project directory set to: {projdir}")

# environ vars
proj_crs = 26913  # NAD83 UTM Zone 13N

In [ ]:
# --- Load the gridMET climatology by Pyrome
gridmet_fp = Path(join(projdir,"data/tabular/raw/weather/gridmet_clim_CO_pyromes.csv"))
# --- Load and inspect
clim = load_gridmet_csv(gridmet_fp) # parses gridmet columns
print(clim.shape)
print("Pyromes :", sorted(clim["pyrome"].unique()))
print("Years   :", sorted(clim["year"].unique()))
print("Columns :", list(clim.columns))

In [ ]:
print("tmmn_f" in clim.columns, "vpd_pa" in clim.columns)

In [ ]:
# --- Build wind scenario by pyrome
from fb_tools.weather import wind_percentiles_from_cell_cache

# --- Build the wind dataframe from HRRR cache
wind_pcts = wind_percentiles_from_cell_cache(
    cache_dir=join(projdir,'data/weather/pyrome_wind'),
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.97],  # default
)
wind_pcts['42']

In [ ]:
# CO pyromes span ~37–41°N; 39.5° used for GSI photoperiod calculation.
# GSI uses tmmn_f (min temp), vpd_pa (from GEE export), and daylength.
# wind_direction=-2 = downhill (worst-case); override with -1 (uphill)
# or an explicit azimuth (0–360) as needed.

CACHE_DIR  = Path(join(projdir,"data/weather/flammap/"))

percentiles = [0.25, 0.50, 0.75, 0.90, 0.97]

scenarios = build_flammap_scenario_cache(
    clim,
    pyrome_col="pyrome",
    percentiles=percentiles,
    out_dir=CACHE_DIR,
    lat_deg=39.5, # adjust as-needed
    wind_direction=-2, # FlamMap downhill default
    wind_percentiles=wind_pcts,
)
scenarios

In [ ]:
# --- Look up Pyrome
# --- Load the Pyrome-specific FlamMap inputs
from fb_tools.weather import load_flammap_scenario_cache

# --- Pyrome 46 CO Rockies
fm_params = load_flammap_scenario_cache(46, CACHE_DIR)
fm_params

In [ ]:
import matplotlib.pyplot as plt

percentile_labels = [f"p{int(p*100)}" for p in fm_params['percentiles']]
fm_herb  = [fm_params['scenarios'][k]['FM_herb']  for k in percentile_labels]
fm_woody = [fm_params['scenarios'][k]['FM_woody'] for k in percentile_labels]
x = fm_params['percentiles']

fig, ax = plt.subplots(figsize=(7, 4))

ax.plot(x, fm_herb,  marker='o', lw=2, color='#2E7D32', label='FM Herb',  zorder=3)
ax.plot(x, fm_woody, marker='s', lw=2, color='#8B4513', label='FM Woody', zorder=3)

# Annotate values
for xi, yh, yw in zip(x, fm_herb, fm_woody):
    ax.annotate(f'{yh:.0f}',  (xi, yh),  textcoords='offset points', xytext=(0, 8),
                ha='center', fontsize=8, color='#2E7D32')
    ax.annotate(f'{yw:.0f}', (xi, yw), textcoords='offset points', xytext=(0, -14),
                ha='center', fontsize=8, color='#8B4513')

ax.set_xticks(x)
ax.set_xticklabels([f"{int(p*100)}th" for p in x])
ax.set_xlabel('ERC Percentile', fontsize=11)
ax.set_ylabel('Fuel Moisture (%)', fontsize=11)
ax.set_title(f"Live Fuel Moisture by ERC Percentile — Pyrome {fm_params['pyrome_id']}", fontsize=12)
ax.legend(framealpha=0.9)
ax.grid(True, linestyle='--', alpha=0.4)
ax.set_ylim(0, max(fm_woody) * 1.15)

plt.tight_layout()
plt.savefig('fm_by_percentile.png', dpi=150, bbox_inches='tight')
plt.show()